# Artifacts and jobs: construct, declare, run, bind

A walkthrough of the artifact machinery against a throwaway folder. Nothing
here touches Modal: every storage call takes a `root`, and an absolute path
on this machine is as good a root as the volume's mount (`config.STORAGE`)
is inside a container.

The shape every family follows: `artifacts/<family>/__init__.py` defines the
artifact class(es); a sibling `jobs.py` defines the job(s) that produce
them. An artifact never imports its own job module, it only *names* it in
`producer`, a dotted-path string. `artifact.job()` is the one call that
turns that string into a class, and with it imports whatever numpy or torch
the family carries. Resolving, declaring and inspecting never reach it.

In [ ]:
import json
import logging
import shutil
from pathlib import Path

import lab
from artifacts.core.artifact import Artifact
from artifacts.core.resolve import resolve
from artifacts.mappeddataset import MappedDataSet
from artifacts.sources import Source
from artifacts.tokenizers.bpe import Tokenizer

# a throwaway local root -- everything below writes only here
ROOT = Path(".scratch/resolve-demo").resolve()
shutil.rmtree(ROOT, ignore_errors=True)
ROOT.mkdir(parents=True)
logging.basicConfig(level=logging.INFO, format="%(message)s", force=True)
ROOT

## Construct

Constructing an artifact is naming its parameters; nothing is written. A
tokenizer's `sources` is a set, so the order you list them in is not part
of what the tokenizer is. `MappedDataSet.from_sources` builds one
`TokenizedSource` per (tokenizer, source) pair, and *its* `train_set` is a
tuple, because concatenation order is part of what the dataset is.

In [ ]:
romeojuliet = Source(name="romeojuliet", url="https://www.gutenberg.org/cache/epub/1513/pg1513.txt")
mobydick = Source(name="mobydick", url="https://www.gutenberg.org/cache/epub/2701/pg2701.txt")

tokenizer = Tokenizer(vocab_size=500, special_tokens=("<|endoftext|>",), sources=(romeojuliet, mobydick))
assert tokenizer == Tokenizer(vocab_size=500, special_tokens=("<|endoftext|>",), sources=(mobydick, romeojuliet))

mapped = MappedDataSet.from_sources(tokenizer, train_sources=[romeojuliet, mobydick], valid_sources=[romeojuliet])

print("artifact_path:", mapped.artifact_path)
print("producer:     ", mapped.producer)  # None: nothing writes it, it is done when its sources are

## Manifest <-> object

`to_manifest()` is the artifact as a dictionary, dependencies embedded as
whole manifests of their own. `"artifact"` is the class's full dotted path,
so reading a manifest back needs no registry. `Artifact.from_manifest` is
the inverse, and always hands back an unbound object; `Artifact.load(path,
root)` is the same round trip read off a `manifest.json` on disk.

In [ ]:
manifest = mapped.to_manifest()
print(json.dumps(manifest, indent=2)[:500], "...")
assert Artifact.from_manifest(manifest) == mapped

## Resolve

`resolve(artifact)` is every artifact the request is built from and then
the request itself, dependencies first, each path once. A shared source
reached through two tokenized sources appears once. It reads nothing.

In [ ]:
for i, artifact in enumerate(resolve(mapped), 1):
    print(f"{i}. {type(artifact).__name__:16} {artifact.artifact_path}")

## Declare

`lab.declare` resolves, compares every path against what `root` holds, and
prints one row per artifact. Without `commit` it writes nothing; with it,
every `new` row gets its manifest, dependencies first, and nothing existing
is ever rewritten. A `conflict` (a different definition already declared at
that path) or an `undeclared` folder (files with no manifest) blocks the
commit, and `verbose=True` shows what differs.

In [ ]:
lab.declare(mapped, root=ROOT)

In [ ]:
report = lab.declare(mapped, root=ROOT, commit=True)

## Run

`artifact.job()` is the one place a `producer` string becomes a class. Running
a job by hand takes a root and a worker; `lab.worker` is the stand-in for one
(no lease, no heartbeat, logs to the console). `status(root)` says which files
are there, so this skips whatever is already built, and a `MappedDataSet` has
no job at all.

In [ ]:
for artifact in resolve(mapped):
    if artifact.producer is None or all(artifact.status(ROOT).completion.values()):
        continue
    print(f"running {artifact.producer} for {artifact.artifact_path}")
    artifact.job().run(ROOT, lab.worker)

lab.declare(mapped, root=ROOT)  # everything reads done now

## Bind

`bind(root)` reads the declaration stored at the artifact's own path, checks
it agrees with the object in hand and that every file is there, and returns a
new object with what those files hold loaded onto it. The object you called it
on stays unbound; reassign.

In [ ]:
mapped = mapped.bind(ROOT)
bound_tokenizer = tokenizer.bind(ROOT)

print(f"{len(mapped.train_tokens)} train tokens across {len(mapped.train_set)} source(s)")
print(repr(bound_tokenizer.decode(mapped.train_tokens[:15])))

In [ ]:
# Cleanup, if you want the demo folder gone:
# shutil.rmtree(ROOT)